# California State Procurement Data — Exploration
Dataset: `PURCHASE ORDER DATA EXTRACT 2012-2015_0.csv`

In [ ]:
import pandas as pd
import numpy as np

CSV_PATH = r"C:\Users\SP7\Downloads\archive (1)\PURCHASE ORDER DATA EXTRACT 2012-2015_0.csv"
df = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")

## 1. Basic Info

In [ ]:
print(f"Total rows    : {len(df):,}")
print(f"Total columns : {len(df.columns)}\n")

print("Column names & dtypes:")
for col in df.columns:
    print(f"  {col:<50} {str(df[col].dtype)}")

In [ ]:
print("Sample — 5 rows:")
df.head(5)

## 2. Data Quality

In [ ]:
# Missing values
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("--- Missing values ---")
if missing.empty:
    print("  None")
else:
    for col, count in missing.items():
        print(f"  {col:<50} {count:>8,}  ({count/len(df)*100:.1f}%)")

# Duplicates
dups = df.duplicated().sum()
print(f"\n--- Duplicate rows ---")
print(f"  {dups:,}  ({dups/len(df)*100:.2f}%)")

In [ ]:
# Date column detection
date_cols = []
print("--- Date columns & sample values ---")
for col in df.columns:
    if df[col].dtype == object:
        sample = df[col].dropna().head(100)
        parsed = pd.to_datetime(sample, infer_datetime_format=True, errors='coerce')
        if parsed.notna().sum() > 80:
            date_cols.append(col)
            print(f"  '{col}' — sample: {df[col].dropna().iloc[0]}")
if not date_cols:
    print("  No date columns detected.")

In [ ]:
# Numeric stored as strings
print("--- Numeric columns stored as strings ---")
numeric_string_cols = []
for col in df.columns:
    if df[col].dtype == object:
        sample = df[col].dropna().head(200).str.replace(r'[\$,]', '', regex=True)
        converted = pd.to_numeric(sample, errors='coerce')
        if converted.notna().sum() / len(sample) > 0.8:
            numeric_string_cols.append(col)
            print(f"  '{col}' — sample: {df[col].dropna().iloc[0]}")
if not numeric_string_cols:
    print("  None found.")

## 3. Key Insights

In [ ]:
# Date range
print("--- Date range ---")
for col in date_cols:
    parsed = pd.to_datetime(df[col], infer_datetime_format=True, errors='coerce')
    print(f"  '{col}': {parsed.min().date()} → {parsed.max().date()}")

In [ ]:
# Unique departments
dept_candidates = [c for c in df.columns if any(k in c.lower() for k in ['dept', 'department'])]
print("--- Unique departments ---")
for col in dept_candidates:
    print(f"  '{col}': {df[col].nunique():,} unique values")
    print("  Top 10:", df[col].value_counts().head(10).to_dict())

In [ ]:
# Unique suppliers
supplier_candidates = [c for c in df.columns if any(k in c.lower() for k in ['supplier', 'vendor', 'contractor', 'company'])]
print("--- Unique suppliers ---")
for col in supplier_candidates:
    print(f"  '{col}': {df[col].nunique():,} unique values")
    print("  Top 10:", df[col].value_counts().head(10).to_dict())

In [ ]:
# Total spending
amount_candidates = [c for c in df.columns if any(k in c.lower() for k in ['amount', 'price', 'cost', 'total', 'spend', 'value'])]
print("--- Total spending per amount column ---")
for col in amount_candidates:
    series = df[col]
    if series.dtype == object:
        series = pd.to_numeric(series.str.replace(r'[\$,]', '', regex=True), errors='coerce')
    print(f"  '{col}': ${series.sum():,.2f}  (non-null rows: {series.notna().sum():,})")

## 4. Analysis for Specific Query Types

In [ ]:
# Orders in a specific time period
print("--- Order date column & monthly distribution ---")
for col in date_cols:
    parsed = pd.to_datetime(df[col], infer_datetime_format=True, errors='coerce')
    print(f"  Column : '{col}'")
    print(f"  Format : {df[col].dropna().iloc[0]}")
    monthly = parsed.dt.to_period('M').value_counts().sort_index()
    print(f"  Monthly order counts (last 6 months in dataset):")
    print(monthly.tail(6).to_string())

In [ ]:
# Quarter with highest spending
print("--- Quarter with highest spending ---")
if date_cols and amount_candidates:
    date_col   = date_cols[0]
    amount_col = amount_candidates[0]
    parsed_dates = pd.to_datetime(df[date_col], infer_datetime_format=True, errors='coerce')
    amounts = df[amount_col]
    if amounts.dtype == object:
        amounts = pd.to_numeric(amounts.str.replace(r'[\$,]', '', regex=True), errors='coerce')
    tmp = pd.DataFrame({'date': parsed_dates, 'amount': amounts})
    tmp['quarter'] = tmp['date'].dt.to_period('Q')
    quarterly = tmp.groupby('quarter')['amount'].sum().sort_values(ascending=False)
    print(f"  Date column   : '{date_col}'")
    print(f"  Amount column : '{amount_col}'")
    print("\n  Top 5 quarters by spending:")
    print(quarterly.head(5).apply(lambda x: f"${x:,.2f}").to_string())

In [ ]:
# Most frequently ordered line items
item_candidates = [c for c in df.columns if any(k in c.lower() for k in ['item', 'description', 'product', 'commodity', 'line'])]
qty_candidates  = [c for c in df.columns if any(k in c.lower() for k in ['qty', 'quantity', 'units'])]
print("--- Most frequently ordered line items ---")
for col in item_candidates[:2]:
    print(f"\n  Column: '{col}'")
    print(df[col].value_counts().head(10).to_string())
print(f"\n  Quantity column: {qty_candidates[0] if qty_candidates else 'Not found'}")
if qty_candidates:
    q = df[qty_candidates[0]]
    if q.dtype == object:
        q = pd.to_numeric(q.str.replace(r'[\$,]', '', regex=True), errors='coerce')
    print(f"  Total quantity ordered: {q.sum():,.0f}")

In [ ]:
# Columns for filtering & grouping
print("--- Columns suitable for filtering & grouping ---")
for col in df.columns:
    if df[col].dtype == object and 2 <= df[col].nunique() <= 500:
        print(f"  '{col}': {df[col].nunique()} unique values")

## 5. Recommendations

In [ ]:
dups = df.duplicated().sum()
missing = df.isnull().sum()
missing = missing[missing > 0]

print("--- Cleaning needed before loading to MongoDB ---")
print("  1. Strip '$' and ',' from monetary columns and cast to float.")
print("  2. Parse date columns to datetime; store as ISO strings or ISODate.")
if dups > 0:
    print(f"  3. Drop {dups:,} duplicate rows.")
if not missing.empty:
    print(f"  4. Handle nulls in: {list(missing.index)}")
print("  5. Normalize column names: lowercase + underscores.")
print("  6. Strip leading/trailing whitespace from all string columns.")

print("\n--- Most important columns ---")
important = date_cols + dept_candidates + supplier_candidates + amount_candidates + item_candidates[:1] + qty_candidates[:1]
seen = []
for c in important:
    if c not in seen:
        seen.append(c)
        print(f"  - {c}")

print("\n--- Potential issues ---")
for col in amount_candidates:
    if df[col].dtype == object:
        print(f"  ! '{col}' is a string — clean before numeric aggregation.")
for col in date_cols:
    parsed = pd.to_datetime(df[col], infer_datetime_format=True, errors='coerce')
    null_dates = parsed.isna().sum()
    if null_dates > 0:
        print(f"  ! '{col}' has {null_dates:,} unparseable date values.")
if dups > 0:
    print(f"  ! {dups:,} duplicate rows may skew aggregations.")
blank_cols = [c for c in df.columns if df[c].dtype == object and df[c].str.strip().eq('').sum() > 0]
if blank_cols:
    print(f"  ! Blank strings (not NaN) found in: {blank_cols}")